# Feature Engineering V3
Enhanced with: competitive matches only, recency weights, H2H, star player ELO, squad age, confederation strength

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Project\FIFA_World_Cup_2026'

# Load all data
competitive = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'results_competitive_v3.csv'))
elo = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'elo_cleaned.csv'))
player_strength = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'player_strength.csv'))
star_elo = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'star_player_elo.csv'))
squad_age = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'squad_age.csv'))
elo_conf = pd.read_csv(os.path.join(BASE, 'Cleaned_Data', 'confederation_strength.csv'))

competitive['date'] = pd.to_datetime(competitive['date'])

print("Competitive matches:", competitive.shape)
print("ELO teams:", elo.shape)
print("Star ELO teams:", star_elo.shape)
print("Squad age teams:", squad_age.shape)

Competitive matches: (5831, 13)
ELO teams: (48, 10)
Star ELO teams: (42, 2)
Squad age teams: (48, 3)


## Lookup Dictionaries

In [3]:
# Build all lookup dicts
elo_lookup = elo.set_index('team')['elo_rating'].to_dict()
elo_wr_lookup = elo.set_index('team')['elo_win_rate'].to_dict()
star_elo_lookup = star_elo.set_index('team')['star_player_elo'].to_dict()
age_lookup = squad_age.set_index('team')['age_score'].to_dict()
conf_lookup = elo_conf.set_index('team')['conf_strength'].to_dict()
ps = player_strength.set_index('team')

def get_player_stat(team, col, default=0):
    try:
        val = ps.loc[team, col]
        return val if not pd.isna(val) else default
    except:
        return default

print("All lookups ready!")

All lookups ready!


## Feature Functions

In [4]:
def get_team_form(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 0.5
    
    wins = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team and row['result'] == 1:
            wins += 1
        elif row['away_team'] == team and row['result'] == -1:
            wins += 1
    
    return wins / len(team_matches)

def get_avg_goals(df, team, date, n=10):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 1.0, 1.0
    
    scored, conceded = [], []
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            scored.append(row['home_score'])
            conceded.append(row['away_score'])
        else:
            scored.append(row['away_score'])
            conceded.append(row['home_score'])
    
    return np.mean(scored), np.mean(conceded)

def get_h2h(df, home, away, date, n=10):
    h2h = df[
        (((df['home_team'] == home) & (df['away_team'] == away)) |
         ((df['home_team'] == away) & (df['away_team'] == home))) &
        (df['date'] < date)
    ].tail(n)
    
    if len(h2h) == 0:
        return 0.5
    
    home_wins = 0
    for _, row in h2h.iterrows():
        if row['home_team'] == home and row['result'] == 1:
            home_wins += 1
        elif row['away_team'] == home and row['result'] == -1:
            home_wins += 1
    
    return home_wins / len(h2h)

def get_winning_momentum(df, team, date, n=5):
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)
    
    if len(team_matches) == 0:
        return 0
    
    consecutive_wins = 0
    for _, row in team_matches.iloc[::-1].iterrows():
        if row['home_team'] == team and row['result'] == 1:
            consecutive_wins += 1
        elif row['away_team'] == team and row['result'] == -1:
            consecutive_wins += 1
        else:
            break
    
    return consecutive_wins / n

print("All functions defined!")

All functions defined!


## Build V3 Feature Matrix

In [5]:
# This will take 3-5 minutes
features = []

for idx, row in competitive.iterrows():
    home = row['home_team']
    away = row['away_team']
    date = row['date']
    
    # Form features
    home_form = get_team_form(competitive, home, date)
    away_form = get_team_form(competitive, away, date)
    
    # Goal features
    home_scored, home_conceded = get_avg_goals(competitive, home, date)
    away_scored, away_conceded = get_avg_goals(competitive, away, date)
    
    # H2H
    h2h = get_h2h(competitive, home, away, date)
    
    # Momentum
    home_momentum = get_winning_momentum(competitive, home, date)
    away_momentum = get_winning_momentum(competitive, away, date)
    
    # ELO features
    home_elo = elo_lookup.get(home, 1500)
    away_elo = elo_lookup.get(away, 1500)
    home_elo_wr = elo_wr_lookup.get(home, 0.5)
    away_elo_wr = elo_wr_lookup.get(away, 0.5)
    
    # Star player ELO
    home_star = star_elo_lookup.get(home, 1500)
    away_star = star_elo_lookup.get(away, 1500)
    
    # Squad age score
    home_age = age_lookup.get(home, 0.5)
    away_age = age_lookup.get(away, 0.5)
    
    # Confederation strength
    home_conf = conf_lookup.get(home, 0.7)
    away_conf = conf_lookup.get(away, 0.7)
    
    # Player strength
    home_attack = get_player_stat(home, 'attack_goals')
    away_attack = get_player_stat(away, 'attack_goals')
    home_defense = get_player_stat(home, 'def_tackles')
    away_defense = get_player_stat(away, 'def_tackles')
    home_mid = get_player_stat(home, 'mid_tackles')
    away_mid = get_player_stat(away, 'mid_tackles')
    
    is_neutral = 1 if row['neutral'] else 0
    
    features.append({
        'date': date,
        'home_team': home,
        'away_team': away,
        # Form
        'home_form': home_form,
        'away_form': away_form,
        'form_diff': home_form - away_form,
        # Goals
        'home_avg_scored': home_scored,
        'home_avg_conceded': home_conceded,
        'away_avg_scored': away_scored,
        'away_avg_conceded': away_conceded,
        'goal_diff': home_scored - away_scored,
        # H2H and momentum
        'h2h': h2h,
        'home_momentum': home_momentum,
        'away_momentum': away_momentum,
        'momentum_diff': home_momentum - away_momentum,
        # ELO
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'home_elo_winrate': home_elo_wr,
        'away_elo_winrate': away_elo_wr,
        # Star players
        'home_star_elo': home_star,
        'away_star_elo': away_star,
        'star_elo_diff': home_star - away_star,
        # Squad age
        'home_age_score': home_age,
        'away_age_score': away_age,
        # Confederation
        'home_conf_strength': home_conf,
        'away_conf_strength': away_conf,
        'conf_diff': home_conf - away_conf,
        # Player strength
        'home_attack': home_attack,
        'away_attack': away_attack,
        'attack_diff': home_attack - away_attack,
        'home_defense': home_defense,
        'away_defense': away_defense,
        'defense_diff': home_defense - away_defense,
        'home_mid': home_mid,
        'away_mid': away_mid,
        # Context
        'is_neutral': is_neutral,
        'weight': row['final_weight'],
        'result': row['result']
    })

features_df = pd.DataFrame(features)
print("V3 Feature matrix shape:", features_df.shape)
features_df.head()

V3 Feature matrix shape: (5831, 39)


,date,home_team,away_team,home_form,away_form,form_diff,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,...,away_attack,attack_diff,home_defense,away_defense,defense_diff,home_mid,away_mid,is_neutral,weight,result
0,2018-01-02,Iraq,United Arab Emirates,0.5,0.5,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,0
1,2018-01-02,Oman,Bahrain,0.5,0.5,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,1
2,2018-01-05,Oman,United Arab Emirates,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,0
3,2018-03-22,Kyrgyzstan,Myanmar,0.5,0.5,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,1
4,2018-03-24,Alderney,Guernsey,0.5,0.5,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,-1


## Save V3 Features

In [6]:
features_df.to_csv(os.path.join(BASE, 'Feature_Engineering', 'features_v3.csv'), index=False)
print("V3 features saved!")
print("Total features:", len(features_df.columns))
print("Shape:", features_df.shape)

V3 features saved!
Total features: 39
Shape: (5831, 39)
